# Telemetry accuracy recalculation

## tl;dr

The start model has two distinct modes. Qualifying uses an IFMAR start: the car waits stationary before start/finish and is called individually. A race uses a stationary grid on this track's back straight, ordered by qualifying. The user identified the final July 11 session (14:49 RaceBox, paired golden/mains data) as the race, so its back-straight grid candidate is selected. The 11:47 and 13:19 sessions remain qualifying-or-practice until distinguished. This confirms current Race Lap 6 = raw lap 7. Its 1.7248 m alignment remains an analysis aid, not surveyed ground truth.

## Context & Methods

This read-only audit discovers unique VBO/RaceBox exports by SHA-256 and excludes duplicate copies. It mirrors the native 25 Hz stationary rule, directed start/finish crossing rejection, and 97-point distance-normalized whole-lap translation.

### Key Assumptions

- **Place down** is the first sustained stationary block before the formation-lap crossing. It can be absent when recording starts with the car already moving.
- **Qualifying start candidate** is the final sustained stationary block before the first accepted start/finish crossing. This models the individual IFMAR start.
- **Race-grid start candidate** is the final sustained stationary block between the first two accepted crossings. It models the stationary grid on this track's back straight after grid formation.
- The final July 11 session is user-labelled **race**, so its race-grid candidate is selected. The 11:47 and 13:19 sessions are user-labelled **qualifying-or-practice** and remain unresolved between those two types.
- Start/finish remains a lap boundary, not the launch location.
- **Race stop** is the final sustained stationary block after the last accepted crossing.
- These landmarks are evaluated within their own session. Cross-run coordinates are incomparable until session type and exact grid/start location are explicitly tagged as matching.
- Google-image calibration proves internal image/trace projection consistency, not surveyed GNSS truth.

In [1]:
import importlib.util, json, pathlib
script = pathlib.Path.cwd().parent / 'scripts' / 'telemetry_accuracy_recalculation.py'
spec = importlib.util.spec_from_file_location('telemetry_accuracy_recalculation', script)
audit = importlib.util.module_from_spec(spec)
spec.loader.exec_module(audit)
results = audit.run_audit()
summary = audit.compact_summary(results)
print(json.dumps(summary, indent=2))

{
  "unique_sessions": 4,
  "sessions": [
    {
      "name": "RaceBox Track Sessionon 03-08-2025 17-25.vbo",
      "samples": 15087,
      "accepted_crossings": 21,
      "metadata_lines": 0,
      "session_type": "unknown",
      "selected_start": null,
      "selected_start_reason": "not selected because the session type is unknown",
      "landmark_detection": {
        "place_down_detected": false,
        "qualifying_start_candidate_detected": false,
        "race_grid_start_candidate_detected": false,
        "race_stop_detected": true,
        "qualifying_start_same_block_as_place_down": false,
        "imu_zero_uses_place_down": false,
        "imu_zero_source": "later_stationary_fallback"
      },
      "place_down": null,
      "qualifying_start_candidate": null,
      "race_grid_start_candidate": null,
      "race_stop": {
        "begin": 14420,
        "end": 15086,
        "seconds": 26.68,
        "lat": 49.18383083333333,
        "lon": -123.14478749999999,
        "ho

## Data & Results

The executed output above is the bounded evidence table. Important interpretation:

- The 2026 two-second IMU-zero windows have 0.155–0.206 m horizontal p95 scatter. Full place-down blocks expand to as much as 1.94 m p95 during a 107.4-second dwell, demonstrating time-dependent GNSS wander within that one stationary landmark.
- Per-session stationary scatter is measurable without assuming shared coordinates. The 14:49 race selects its back-straight grid block at samples 4045–4420 (15.04 s stationary, 0.332 m horizontal p95).
- Qualifying-start and race-grid candidates remain visible for 11:47 and 13:19, but neither is promoted until those sessions are separated into qualifying versus practice.
- The 2025 recording starts with the car moving, so neither place-down, IFMAR qualifying-start, nor race-grid-start landmarks can be recovered automatically. Its current IMU zero comes from a later stop at 96.16 s and must be disclosed as a fallback, not called the initial on-track zero.
- 2026 IMU stationary medians are repeatable: longitudinal −0.014 to −0.019 g, lateral +0.051 to +0.059 g, vertical +0.992 to +0.994 g; gyro Y remains −0.55 deg/s. The 2025 mounting/zero differs, so per-session calibration is necessary.
- R6/raw 7 correction is almost entirely northward (east −0.042 m, north −1.724 m). Against adjacent raw laps it still needs 0.989–1.615 m correction, with 1.20–1.26 m RMS residual.
- The untouched 11:47 CSV export contains 12 metadata lines before the actual data header. The current native CSV parser expects the header on line 1; direct loading of that raw download is therefore not yet reliable.

## Takeaways

1. Keep the visible R6 translation as an explicitly derived comparison aid; do not call it corrected ground truth.
2. Confidence should use each session's own stationary scatter, translation magnitude, residual shape error, satellite count, and cross-lap repeatability. Never use unmatched start/grid/stop coordinates as cross-session error.
3. Persist the known 14:49 race label and back-straight grid start. Add explicit practice/qualifying labels for 11:47 and 13:19 before selecting their starts.
4. Fix raw RaceBox CSV metadata-header ingestion before using the additional sessions inside the app.
5. Stabilize automatic corner numbering across reference laps; current native checks return both 9 and 11 corners depending on the reference.
6. Preserve conservative insight wording: the evidence engine currently finds no reliable technique recommendation in the golden session, which is safer than presenting consequential gains as causal advice.
7. For verified sub-metre truth, add surveyed control points or RTK. These logs measure repeatability, not absolute accuracy.